# Defensive Cyber 135M: full-weight adaptation

This run starts from the Apache-2.0 SmolLM2-135M-Instruct model, which already has English fluency and instruction behavior. It full-fine-tunes all model weights on a filtered defensive corpus. It does not train a 135M model from random initialization. The holdout is kept separate.

The run prints optimizer step, data epoch fraction, assistant-token loss, tokens/second, steps/second, elapsed time, and ETA. MAE is not applicable to next-token prediction.

In [ ]:
import glob, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No Kaggle GPU detected. Select an available GPU accelerator and restart the session.')
print('GPU:', torch.cuda.get_device_name(0), '| PyTorch:', torch.__version__, flush=True)
try:
    import transformers, datasets
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.48,<5', 'datasets>=3.2,<5', 'accelerate>=1.2', 'safetensors>=0.4'], check=True)
    import transformers, datasets
print('Transformers:', transformers.__version__, '| Datasets:', datasets.__version__, flush=True)
work = Path('/kaggle/working/cyber_135m')
data_in = work / 'bundle_data'
clean_data = work / 'clean_data'
output = Path('/kaggle/working/output')
work.mkdir(parents=True, exist_ok=True); data_in.mkdir(parents=True, exist_ok=True); output.mkdir(parents=True, exist_ok=True)
bundle = work / 'input_bundle'; bundle.mkdir(parents=True, exist_ok=True)
for pattern in ('cyber_135m.zip', 'data.zip'):
    matches = glob.glob('/kaggle/input/**/' + pattern, recursive=True)
    for archive in matches:
        with zipfile.ZipFile(archive) as zf: zf.extractall(bundle)
# Kaggle may mount directory uploads as extracted paths or as zip files; accept both layouts.
scripts = glob.glob(str(bundle / 'cyber_135m' / 'prepare_sft.py')) + glob.glob('/kaggle/input/**/cyber_135m/prepare_sft.py', recursive=True)
if not scripts: raise FileNotFoundError('The attached private data bundle is missing cyber_135m/prepare_sft.py.')
bundle = Path(scripts[0]).parents[1]
shutil.copytree(bundle / 'cyber_135m', work, dirs_exist_ok=True)
if (bundle / 'data').exists(): shutil.copytree(bundle / 'data', data_in, dirs_exist_ok=True)
print('Input bundle copied. Raw Copilot conversation rows are not attached.', flush=True)

In [ ]:
print('Starting data audit and Fenrir download.', flush=True)
subprocess.run([sys.executable, '-u', str(work / 'prepare_sft.py'), '--bundle', str(data_in), '--output-dir', str(clean_data), '--include-fenrir', '--guardrails', str(work / 'guardrails.md')], check=True, cwd=str(work))
print('Prepared audit:', (clean_data / 'data_audit.json').read_text(), flush=True)

In [ ]:
print('Starting full-weight fine-tuning; progress is printed every optimizer step.', flush=True)
subprocess.run([sys.executable, '-u', str(work / 'train_sft.py'), '--train', str(clean_data / 'train.jsonl'), '--validation', str(clean_data / 'validation.jsonl'), '--out-dir', str(output), '--max-length', '768', '--batch-size', '2', '--grad-accum', '4', '--epochs', '1.0', '--max-steps', '12000', '--eval-interval', '500', '--patience', '6', '--learning-rate', '2e-5'], check=True, cwd=str(work))

In [ ]:
print('Generating fixed prompts from the base and tuned models for side-by-side review.', flush=True)
subprocess.run([sys.executable, '-u', str(work / 'evaluate_samples.py'), '--adapted', str(output / 'best_model'), '--output', str(output / 'generation_comparison.csv')], check=True, cwd=str(work))
print('Scoring the untouched expert-reviewed QAA holdout before and after adaptation.', flush=True)
subprocess.run([sys.executable, '-u', str(work / 'evaluate_holdout.py'), '--eval-file', str(data_in / 'cybersecurity_qa_eval.jsonl'), '--adapted', str(output / 'best_model'), '--output-dir', str(output), '--sample-count', '12'], check=True, cwd=str(work))
print('Run summary:', (output / 'run_summary.json').read_text(), flush=True)
print('Output files:', sorted(p.name for p in output.iterdir()), flush=True)